[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/B.%20Core%20Quay-Side%20Operations%20%28The%20Ship-to-Shore%20Interface%29/08.%20The%20Quay%20Crane%20Assignment%20Problem/P8-Tier-4_executed.ipynb)

---

# 8. The Quay Crane Assignment Problem
## Tier 4 — Transfer Learning for QCAP-Related Prediction

### Goal
This notebook demonstrates a transfer-learning workflow using synthetic data inspired by the Quay Crane Assignment Problem (QCAP). A neural network is trained on instances from several source ports. Its learned weights and biases then initialize a model that is fine-tuned using a smaller set of instances from a target port. A model trained from scratch on the same target training split provides a reference for comparison.

The prediction target is a **synthetic aggregate vessel-processing-time measure** produced by a stated greedy crane-assignment rule. It is not an exact QCAP optimum, and this notebook does not solve the QCAP optimization problem. The neural network predicts the measure from instance features; it does not choose a crane assignment.

### Inputs, outputs, and workflow
- **Inputs:** Synthetic vessel workloads, vessel–crane productivity rates, and a port-level interference factor.
- **Training labels:** Aggregate vessel-processing time calculated from the greedy assignment.
- **Outputs:** Source-domain, target-validation, and held-out target-test prediction errors, training times, and plots.

The workflow is:
1. Generate synthetic source, target-training, target-validation, and target-test data.
2. Fit a feature scaler on source training data and train a multilayer perceptron on the source domain.
3. Initialize a target-domain model with the trained source model's weights and biases, then fine-tune it on target training data.
4. Compare the fine-tuned model with a model trained from scratch, and evaluate both on held-out target test data.

### Model assumptions and scope
The source ports are Rotterdam, Hamburg, and Antwerp; the target port is Singapore. Port sizes differ, so each instance uses a common padded feature layout. Zero padding represents vessels or cranes that do not exist in a particular port. The greedy assignment gives each vessel one crane, then assigns a second crane to vessels in descending workload order while cranes remain. At each assignment step, the available crane with the highest productivity for the vessel is selected. The assignment does not model crane travel, vessel geometry, quay interference between vessels, precedence constraints, operational uncertainty, or an exact optimization-based assignment. The interference factor reduces the combined productivity of cranes assigned to the same vessel.

All data are synthetic. The stated sample counts describe generated instances, not observed port operations. The held-out test set is used only for final evaluation. Transfer learning is implemented by copying the trained source network's weights and biases into a compatible target model before fine-tuning.

## Step 1: Problem Setup and Data Generation

The first step creates comparable training examples for ports with different numbers of vessels and cranes. A neural network requires every example to have the same number of input features, so workload and crane-productivity arrays are padded to the largest port dimensions in this example.

For an instance with vessel set $V$, each vessel $v$ has workload $w_v$ in TEU, and each vessel–crane pair $(v,c)$ has productivity $p_{vc}$ in TEU per hour. Let $C_v$ be the set of cranes assigned to vessel $v$, and let $k_v = |C_v|$. The synthetic rate and processing-time label are defined by

$$
P_v = (1 - \alpha)^{k_v - 1} \sum_{c \in C_v} p_{vc},
\qquad
T_v = \frac{w_v}{P_v},
\qquad
T_{\mathrm{agg}} = \sum_{v \in V} T_v.
$$

Here, $\alpha$ is the port's interference factor, $P_v$ is the effective productivity assigned to vessel $v$, $T_v$ is that vessel's processing time under the assumed rate model, and $T_{\mathrm{agg}}$ is the sum of vessel processing times used as the regression label. The exponent makes the interference penalty apply once for each crane beyond the first. This sum is an aggregate measure; it is not the elapsed time until all vessels finish in parallel.

The greedy label generator first considers vessels in descending workload order and gives each vessel one available crane, choosing the crane with the highest productivity for that vessel. It then considers vessels in the same order and gives a second crane to each vessel that has fewer than two cranes, until no cranes remain. Every crane is used at most once. This rule generates labels; it is not an exact QCAP solver.

Each feature vector concatenates padded vessel workloads, the flattened padded vessel–crane productivity matrix, the interference factor, and the number of assigned cranes for each vessel. Stable instance identifiers make the synthetic data reproducible across Python sessions.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Tuple
import time
import zlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler


@dataclass
class PortData:
    """Synthetic configuration for a port domain."""

    name: str
    num_vessels: int
    num_cranes: int
    workload_range: Tuple[int, int]
    productivity_base: float
    interference_factor: float


source_ports = [
    PortData("Rotterdam", 5, 8, (1500, 2500), 1.0, 0.08),
    PortData("Hamburg", 4, 6, (1200, 2000), 0.95, 0.09),
    PortData("Antwerp", 6, 9, (1000, 1800), 0.98, 0.07),
]

target_port = PortData("Singapore", 5, 8, (1800, 2800), 1.05, 0.06)

SOURCE_SAMPLES = 15000
TARGET_SAMPLES = 800
TEST_SAMPLES = 200

MAX_VESSELS = max(
    port.num_vessels for port in source_ports + [target_port]
)
MAX_CRANES = max(
    port.num_cranes for port in source_ports + [target_port]
)

print("Transfer Learning Setup:")
print(f"Source domains: {[port.name for port in source_ports]}")
print(f"Target domain: {target_port.name}")
print(f"Source samples: {SOURCE_SAMPLES}, target samples: {TARGET_SAMPLES}")
print(f"Independent target test samples: {TEST_SAMPLES}")
print(f"Padded feature-vector length: {MAX_VESSELS + MAX_VESSELS * MAX_CRANES + 1 + MAX_VESSELS}")


def greedy_crane_assignment(
    workloads: np.ndarray,
    productivity: np.ndarray,
    interference: float,
) -> Dict[int, List[int]]:
    """Assign one crane to each vessel, then greedily add a second where possible."""
    num_vessels, num_cranes = productivity.shape
    assignment = {vessel: [] for vessel in range(num_vessels)}
    used_cranes = set()
    vessel_order = np.argsort(workloads)[::-1]

    # Giving every vessel one crane keeps every workload represented in the label.
    for vessel_idx in vessel_order:
        available_cranes = [
            crane for crane in range(num_cranes) if crane not in used_cranes
        ]
        if not available_cranes:
            break
        crane_idx = max(
            available_cranes,
            key=lambda crane: productivity[vessel_idx, crane],
        )
        assignment[vessel_idx].append(crane_idx)
        used_cranes.add(crane_idx)

    # Remaining cranes are assigned in workload order, with at most two per vessel.
    for vessel_idx in vessel_order:
        if len(used_cranes) == num_cranes:
            break
        if len(assignment[vessel_idx]) >= 2:
            continue
        available_cranes = [
            crane for crane in range(num_cranes) if crane not in used_cranes
        ]
        if available_cranes:
            crane_idx = max(
                available_cranes,
                key=lambda crane: productivity[vessel_idx, crane],
            )
            assignment[vessel_idx].append(crane_idx)
            used_cranes.add(crane_idx)

    return assignment


def calculate_assignment_time(
    workloads: np.ndarray,
    productivity: np.ndarray,
    assignment: Dict[int, List[int]],
    interference: float,
) -> float:
    """Return the sum of vessel processing times under the assumed rate model."""
    total_time = 0.0

    for vessel_idx, cranes in assignment.items():
        if not cranes:
            raise ValueError(f"Vessel {vessel_idx} has no assigned crane.")

        num_cranes = len(cranes)
        penalty = (1 - interference) ** (num_cranes - 1)
        effective_productivity = float(
            np.sum(productivity[vessel_idx, cranes]) * penalty
        )
        if effective_productivity <= 0:
            raise ValueError("Effective productivity must be positive.")
        total_time += workloads[vessel_idx] / effective_productivity

    return total_time


def generate_qcap_instance(
    port: PortData,
    instance_id: int,
) -> Tuple[np.ndarray, float]:
    """Generate padded features and a greedy-assignment aggregate-time label."""
    rng = np.random.default_rng(instance_id)
    vessel_workloads = rng.uniform(
        port.workload_range[0],
        port.workload_range[1],
        port.num_vessels,
    )

    base_productivity = port.productivity_base * 25
    productivity_matrix = rng.normal(
        base_productivity,
        base_productivity * 0.1,
        (port.num_vessels, port.num_cranes),
    )
    productivity_matrix = np.maximum(
        productivity_matrix,
        base_productivity * 0.8,
    )

    assignment = greedy_crane_assignment(
        vessel_workloads,
        productivity_matrix,
        port.interference_factor,
    )
    aggregate_time = calculate_assignment_time(
        vessel_workloads,
        productivity_matrix,
        assignment,
        port.interference_factor,
    )

    # Padding gives every port the same feature count while preserving its shape.
    padded_workloads = np.zeros(MAX_VESSELS)
    padded_workloads[:port.num_vessels] = vessel_workloads

    padded_productivity = np.zeros((MAX_VESSELS, MAX_CRANES))
    padded_productivity[:port.num_vessels, :port.num_cranes] = productivity_matrix

    crane_counts = np.zeros(MAX_VESSELS)
    for vessel_idx, cranes in assignment.items():
        crane_counts[vessel_idx] = len(cranes)

    features = np.concatenate(
        [
            padded_workloads,
            padded_productivity.ravel(),
            [port.interference_factor],
            crane_counts,
        ]
    )
    return features, aggregate_time


def stable_instance_id(domain: str, index: int) -> int:
    """Create a stable nonnegative seed without relying on randomized hash()."""
    key = f"{domain}:{index}".encode("utf-8")
    return zlib.crc32(key)


source_data = []
samples_per_port = SOURCE_SAMPLES // len(source_ports)

print("\nGenerating source-domain data...")
for port in source_ports:
    port_data = []
    for index in range(samples_per_port):
        features, aggregate_time = generate_qcap_instance(
            port,
            stable_instance_id(port.name, index),
        )
        port_data.append(
            {
                "port": port.name,
                "features": features,
                "completion_time": aggregate_time,
            }
        )
    source_data.extend(port_data)
    print(f"  {port.name}: {len(port_data)} instances")


def generate_domain_data(port: PortData, count: int, domain: str) -> pd.DataFrame:
    """Generate a reproducible set of instances for one domain."""
    rows = []
    for index in range(count):
        features, aggregate_time = generate_qcap_instance(
            port,
            stable_instance_id(domain, index),
        )
        rows.append(
            {
                "port": port.name,
                "features": features,
                "completion_time": aggregate_time,
            }
        )
    return pd.DataFrame(rows)


source_df = pd.DataFrame(source_data)
target_df = generate_domain_data(target_port, TARGET_SAMPLES, "Singapore-target")
test_df = generate_domain_data(target_port, TEST_SAMPLES, "Singapore-test")

print(f"\nTarget-domain training pool: {len(target_df)} instances")
print(f"Independent target test set: {len(test_df)} instances")
print("\nSynthetic label summary:")
print(
    "Source aggregate processing time: "
    f"{source_df['completion_time'].mean():.2f} ± "
    f"{source_df['completion_time'].std():.2f} hours"
)
print(
    "Target aggregate processing time: "
    f"{target_df['completion_time'].mean():.2f} ± "
    f"{target_df['completion_time'].std():.2f} hours"
)
print(f"Feature vector length: {len(source_df['features'].iloc[0])}")

assert len(source_df) == samples_per_port * len(source_ports)
assert len(source_df['features'].iloc[0]) == len(target_df['features'].iloc[0])
assert all(
    len(features) == len(source_df['features'].iloc[0])
    for features in test_df["features"]
)
print("Data generation and feature-shape checks completed.")

## Step 2: Base Model Training on Source Domains

A multilayer perceptron (MLP) maps an input feature vector through successive layers of weighted sums and nonlinear activation functions. For an input vector $x$, its prediction can be written as $f_{\theta}(x)$, where $\theta$ collects the network's weights and biases. Here, the network learns a regression mapping from padded instance features to the synthetic aggregate processing-time label.

The scaler converts each feature using statistics learned from source training data. For feature $j$, the transformed value is $z_j = (x_j - \mu_j) / s_j$, where $\mu_j$ and $s_j$ are the source-training mean and scale. Applying the same transformation to validation and target data keeps feature units comparable and prevents validation or test examples from influencing the scaling parameters.

The target-domain training pool is split at this stage. The same target training and validation examples are then used for the transfer and from-scratch models, making their validation comparison use a shared data split.

In [ ]:
X_source = np.vstack(source_df["features"].to_numpy())
y_source = source_df["completion_time"].to_numpy()

X_source_train, X_source_val, y_source_train, y_source_val = train_test_split(
    X_source,
    y_source,
    test_size=0.2,
    random_state=42,
)

X_target = np.vstack(target_df["features"].to_numpy())
y_target = target_df["completion_time"].to_numpy()
(
    X_target_train,
    X_target_val,
    y_target_train,
    y_target_val,
) = train_test_split(
    X_target,
    y_target,
    test_size=0.2,
    random_state=42,
)

scaler = StandardScaler()
X_source_train_scaled = scaler.fit_transform(X_source_train)
X_source_val_scaled = scaler.transform(X_source_val)
X_target_train_scaled = scaler.transform(X_target_train)
X_target_val_scaled = scaler.transform(X_target_val)

base_model = MLPRegressor(
    hidden_layer_sizes=(128, 64, 32),
    activation="relu",
    solver="adam",
    alpha=0.001,
    batch_size=64,
    learning_rate="adaptive",
    max_iter=200,
    random_state=42,
    verbose=True,
)

print("Training the base model on source-domain data...")
start_time = time.time()
base_model.fit(X_source_train_scaled, y_source_train)
source_training_time = time.time() - start_time

y_source_pred = base_model.predict(X_source_val_scaled)
source_mae = mean_absolute_error(y_source_val, y_source_pred)
source_rmse = np.sqrt(mean_squared_error(y_source_val, y_source_pred))

print("\nBase-model performance on source validation data:")
print(f"  Training time: {source_training_time:.2f} seconds")
print(f"  MAE: {source_mae:.2f} hours")
print(f"  RMSE: {source_rmse:.2f} hours")
print(f"  Mean prediction error: {np.mean(y_source_pred - y_source_val):.2f} hours")

# This validation comparison describes domain shift; it does not use the final test set.
y_target_pred_base = base_model.predict(X_target_val_scaled)
target_mae_base = mean_absolute_error(y_target_val, y_target_pred_base)
target_rmse_base = np.sqrt(mean_squared_error(y_target_val, y_target_pred_base))

print("\nSource-trained model on target validation data (before adaptation):")
print(f"  MAE: {target_mae_base:.2f} hours")
print(f"  RMSE: {target_rmse_base:.2f} hours")
print(
    "  Mean prediction error: "
    f"{np.mean(y_target_pred_base - y_target_val):.2f} hours"
)

errors_source = y_source_pred - y_source_val
errors_target_base = y_target_pred_base - y_target_val

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].scatter(y_source_val, y_source_pred, alpha=0.6, color="blue")
source_limits = [y_source_val.min(), y_source_val.max()]
axes[0].plot(source_limits, source_limits, "r--", linewidth=2, label="Perfect prediction")
axes[0].set_xlabel("Actual aggregate processing time (hours)")
axes[0].set_ylabel("Predicted aggregate processing time (hours)")
axes[0].set_title("Source-domain validation")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].scatter(y_target_val, y_target_pred_base, alpha=0.6, color="orange")
target_limits = [y_target_val.min(), y_target_val.max()]
axes[1].plot(target_limits, target_limits, "r--", linewidth=2, label="Perfect prediction")
axes[1].set_xlabel("Actual aggregate processing time (hours)")
axes[1].set_ylabel("Predicted aggregate processing time (hours)")
axes[1].set_title("Target validation before adaptation")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].hist(errors_source, bins=30, alpha=0.7, label="Source validation", color="blue")
axes[2].hist(
    errors_target_base,
    bins=30,
    alpha=0.7,
    label="Target validation before adaptation",
    color="orange",
)
axes[2].set_xlabel("Prediction error (hours; prediction − actual)")
axes[2].set_ylabel("Frequency")
axes[2].set_title("Validation error distributions")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nValidation error spread and mean shift:")
print(f"  Source error standard deviation: {np.std(errors_source):.2f} hours")
print(f"  Target error standard deviation: {np.std(errors_target_base):.2f} hours")
print(
    "  Absolute difference between mean errors: "
    f"{abs(np.mean(errors_target_base) - np.mean(errors_source)):.2f} hours"
)

## Step 3: Transfer-Learning Adaptation

Transfer learning uses parameters learned in a source task to initialize a model for a related target task. In this notebook, the source and target networks have the same input layout and architecture, so corresponding layer weights and biases have matching dimensions.

Write the source parameters as $\theta_{\mathrm{source}} = (W_1, b_1, \ldots, W_L, b_L)$, where $W_\ell$ and $b_\ell$ are the weight matrix and bias vector for layer $\ell$. The target model starts from $\theta_{\mathrm{target}}^{(0)} = \theta_{\mathrm{source}}$ and is then fitted using target training examples. Fine-tuning updates these initialized parameters to reduce target-domain prediction error. The from-scratch reference instead begins with its own random initialization.

The `warm_start=True` setting tells scikit-learn to retain the initialized network parameters when fitting. The copied arrays contain the learned weights and biases; the model also needs the fitted network structure, including its layer count, to use those arrays during training. Both models are evaluated on the same target validation split. The validation data monitor adaptation; the independent test set remains unused until Step 4.

Training time and validation error depend on the software environment and optimization run, so they are measured when the notebook is executed.

In [ ]:
transfer_model = MLPRegressor(
    hidden_layer_sizes=(128, 64, 32),
    activation="relu",
    solver="adam",
    alpha=0.001,
    batch_size=32,
    learning_rate="adaptive",
    learning_rate_init=0.001,
    max_iter=100,
    random_state=42,
    warm_start=True,
)

# Matching network shapes are required because each transferred matrix maps between specific layers.
assert transfer_model.hidden_layer_sizes == base_model.hidden_layer_sizes
assert base_model.coefs_[0].shape[0] == X_target_train_scaled.shape[1]
assert base_model.coefs_[-1].shape[1] == 1

# The fitted layer metadata lets scikit-learn traverse the transferred network during fine-tuning.
transfer_model.coefs_ = [weights.copy() for weights in base_model.coefs_]
transfer_model.intercepts_ = [biases.copy() for biases in base_model.intercepts_]
transfer_model.n_layers_ = base_model.n_layers_
transfer_model.n_features_in_ = base_model.n_features_in_
transfer_model.n_iter_ = base_model.n_iter_
transfer_model.t_ = base_model.t_
transfer_model.n_outputs_ = base_model.n_outputs_
transfer_model.out_activation_ = base_model.out_activation_

print("Fine-tuning the source-initialized model on target training data...")
start_time = time.time()
transfer_model.fit(X_target_train_scaled, y_target_train)
transfer_training_time = time.time() - start_time

y_transfer_pred = transfer_model.predict(X_target_val_scaled)
transfer_mae = mean_absolute_error(y_target_val, y_transfer_pred)
transfer_rmse = np.sqrt(mean_squared_error(y_target_val, y_transfer_pred))

scratch_model = MLPRegressor(
    hidden_layer_sizes=(128, 64, 32),
    activation="relu",
    solver="adam",
    alpha=0.001,
    batch_size=32,
    learning_rate="adaptive",
    max_iter=100,
    random_state=42,
)

print("Training the from-scratch reference on target training data...")
start_time = time.time()
scratch_model.fit(X_target_train_scaled, y_target_train)
scratch_training_time = time.time() - start_time

y_scratch_pred = scratch_model.predict(X_target_val_scaled)
scratch_mae = mean_absolute_error(y_target_val, y_scratch_pred)
scratch_rmse = np.sqrt(mean_squared_error(y_target_val, y_scratch_pred))

results_comparison = pd.DataFrame(
    {
        "Method": ["From scratch", "Transfer learning"],
        "Training time (seconds)": [scratch_training_time, transfer_training_time],
        "Validation MAE (hours)": [scratch_mae, transfer_mae],
        "Validation RMSE (hours)": [scratch_rmse, transfer_rmse],
    }
)

print("\nTarget-validation comparison:")
print(results_comparison.round(3).to_string(index=False))

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
methods = ["From scratch", "Transfer learning"]
colors = ["red", "green"]

axes[0, 0].bar(methods, [scratch_training_time, transfer_training_time], color=colors, alpha=0.7)
axes[0, 0].set_ylabel("Training time (seconds)")
axes[0, 0].set_title("Target training time")
axes[0, 0].grid(True, axis="y", alpha=0.3)

axes[0, 1].bar(methods, [scratch_mae, transfer_mae], color=colors, alpha=0.7)
axes[0, 1].set_ylabel("Mean absolute error (hours)")
axes[0, 1].set_title("Target validation MAE")
axes[0, 1].grid(True, axis="y", alpha=0.3)

axes[1, 0].bar(methods, [scratch_rmse, transfer_rmse], color=colors, alpha=0.7)
axes[1, 0].set_ylabel("Root mean squared error (hours)")
axes[1, 0].set_title("Target validation RMSE")
axes[1, 0].grid(True, axis="y", alpha=0.3)

axes[1, 1].scatter(
    y_target_val,
    y_scratch_pred,
    alpha=0.6,
    color="red",
    label="From scratch",
)
axes[1, 1].scatter(
    y_target_val,
    y_transfer_pred,
    alpha=0.6,
    color="green",
    label="Transfer learning",
)
validation_limits = [y_target_val.min(), y_target_val.max()]
axes[1, 1].plot(
    validation_limits,
    validation_limits,
    "k--",
    linewidth=2,
    label="Perfect prediction",
)
axes[1, 1].set_xlabel("Actual aggregate processing time (hours)")
axes[1, 1].set_ylabel("Predicted aggregate processing time (hours)")
axes[1, 1].set_title("Target validation predictions")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  Lower MAE or RMSE indicates smaller prediction errors on this validation split.")
print("  A shorter measured fit time is not, by itself, evidence of better prediction quality.")
print("  The comparison describes this synthetic sample and one training run.")

## Step 4: Final Evaluation on Held-Out Target Data

The target test instances have separate, reproducibly generated identifiers and were not used to fit the scaler or either model. This separation makes the test metrics a final assessment of generalization to new synthetic instances from the target domain.

Mean absolute error (MAE) is the average absolute difference between predicted and actual aggregate processing time. Root mean squared error (RMSE) gives greater weight to larger errors. Mean error retains the sign: a positive value indicates average overprediction, while a negative value indicates average underprediction. For observed labels $y_i$ and predictions $\hat{y}_i$, these metrics are

$$
\mathrm{MAE} = \frac{1}{n}\sum_{i=1}^{n}|\hat{y}_i-y_i|,
\qquad
\mathrm{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(\hat{y}_i-y_i)^2},
\qquad
\mathrm{MeanError} = \frac{1}{n}\sum_{i=1}^{n}(\hat{y}_i-y_i).
$$

Here, $n$ is the number of test instances. Prediction error is not automatically operational time saved. Translating predictive accuracy into vessel capacity, delay reduction, or financial benefit would require an operational decision model and validated real-world data, neither of which is part of this example.

In [ ]:
X_test = np.vstack(test_df["features"].to_numpy())
y_test = test_df["completion_time"].to_numpy()
X_test_scaled = scaler.transform(X_test)

y_test_transfer = transfer_model.predict(X_test_scaled)
y_test_scratch = scratch_model.predict(X_test_scaled)

test_transfer_mae = mean_absolute_error(y_test, y_test_transfer)
test_transfer_rmse = np.sqrt(mean_squared_error(y_test, y_test_transfer))
test_scratch_mae = mean_absolute_error(y_test, y_test_scratch)
test_scratch_rmse = np.sqrt(mean_squared_error(y_test, y_test_scratch))

final_results = pd.DataFrame(
    {
        "Method": ["From scratch", "Transfer learning"],
        "Test MAE (hours)": [test_scratch_mae, test_transfer_mae],
        "Test RMSE (hours)": [test_scratch_rmse, test_transfer_rmse],
        "Mean error (hours)": [
            np.mean(y_test_scratch - y_test),
            np.mean(y_test_transfer - y_test),
        ],
    }
)

print("Final evaluation on the independent target test set:")
print(final_results.round(3).to_string(index=False))

if test_scratch_mae > 0:
    mae_change_percent = (
        (test_scratch_mae - test_transfer_mae) / test_scratch_mae * 100
    )
    print(
        "\nTransfer-learning MAE reduction relative to the scratch model: "
        f"{mae_change_percent:.1f}%"
    )
else:
    print("\nThe scratch-model test MAE is zero; a relative MAE change is undefined.")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(
    y_test,
    y_test_scratch,
    alpha=0.6,
    color="red",
    label="From scratch",
)
axes[0].scatter(
    y_test,
    y_test_transfer,
    alpha=0.6,
    color="green",
    label="Transfer learning",
)
test_limits = [y_test.min(), y_test.max()]
axes[0].plot(
    test_limits,
    test_limits,
    "k--",
    linewidth=2,
    label="Perfect prediction",
)
axes[0].set_xlabel("Actual aggregate processing time (hours)")
axes[0].set_ylabel("Predicted aggregate processing time (hours)")
axes[0].set_title("Held-out target test predictions")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

errors_scratch_test = y_test_scratch - y_test
errors_transfer_test = y_test_transfer - y_test
axes[1].boxplot(
    [errors_scratch_test, errors_transfer_test],
    labels=["From scratch", "Transfer learning"],
)
axes[1].axhline(0, color="black", linestyle="--", linewidth=1)
axes[1].set_ylabel("Prediction error (hours; prediction − actual)")
axes[1].set_title("Held-out test error distributions")
axes[1].grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("\nEvaluation checks:")
assert len(y_test) == TEST_SAMPLES
assert np.isfinite(y_test_transfer).all()
assert np.isfinite(y_test_scratch).all()
print("  Both models produced finite predictions for every held-out test instance.")

## Results Interpretation and Key Insights

### What this experiment can show
- **Domain adaptation:** The source-trained model's target-validation performance can be compared with its source-validation performance to examine how the synthetic domain shift affects prediction.
- **Weight transfer:** The target model begins with learned source-domain weights and biases, then updates them using target training examples.
- **Data-efficiency question:** The experiment uses a smaller target-domain training pool than the source-domain pool. Whether this is sufficient is determined by the validation and held-out test metrics; it is not guaranteed by the method.
- **Model comparison:** The scratch and transfer models use the same target training and validation splits. Differences in their measured performance are specific to these data, model settings, and training runs.

### Limitations
- The labels come from a simplified greedy assignment and a synthetic productivity model; they are not optimal QCAP solutions or measured terminal outcomes.
- The target test set is independent of model fitting, but all data follow the same synthetic generation assumptions. Good test performance therefore does not establish real-port performance.
- One run and one fixed split do not measure variability across data samples or neural-network initializations.
- Predicting an aggregate processing-time measure is different from selecting crane assignments or demonstrating operational savings.

### When this workflow may be relevant
Transfer learning can be considered when a new terminal has limited labeled data and there are related source domains, provided that the feature representation and prediction target are compatible. Its usefulness should be established with appropriately separated validation and test data and, for operational use, validated against real terminal data.